# Whale Alert + Price Analysis (from merged table)

Data source: `data/cleaned/cleaned_1m_with_whale.csv`

This notebook uses the merged minute table only, then:
1. Aggregates to 15-minute bars for candlestick + alert analysis.
2. Validates alert behavior by volatility regime (`low/mid/high`); **the vol regime tables use only the time range from the first whale alert minute to the last** (rolling vol still uses full history where needed). **Mean/median/p90 of alerts use only bars with alert activity (`> 0`),** plus `n_nonzero` per regime.
3. Plots daily alert activity and daily notional alert amount on full whale-data days only.

## NEW VERSION (Merged Table Driven)

下面开始是重写后的完整流程（使用 `data/cleaned/cleaned_1m_with_whale.csv` 作为唯一数据源）。

> 建议从这一节开始 `Run All`。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

MERGED_PATH = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale.csv"

df = pd.read_csv(MERGED_PATH)
df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce").dt.floor("min")
if "minute_utc" in df.columns:
    df["minute_utc"] = pd.to_datetime(df["minute_utc"], utc=True, errors="coerce").dt.floor("min")

num_cols = ["open", "high", "low", "close", "volume", "whale_alert_count", "whale_alert_notional_abs"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["time_utc", "open", "high", "low", "close"]).sort_values("time_utc")

print("Loaded:", MERGED_PATH)
print("Rows:", len(df), "Cols:", df.shape[1])
print("Range:", df["time_utc"].min(), "->", df["time_utc"].max())

## NEW VERSION
Use merged table only.

In [ ]:
# Build 15m OHLC and 15m alert from merged 1m table
START_UTC = pd.to_datetime(df["minute_utc"].dropna().min(), utc=True, errors="coerce")
if pd.isna(START_UTC):
    START_UTC = df["time_utc"].min()

price_15m = (
    df.set_index("time_utc")[["open", "high", "low", "close"]]
    .resample("15min")
    .agg({"open": "first", "high": "max", "low": "min", "close": "last"})
    .dropna()
)

alert_15m = (
    df.set_index("time_utc")["whale_alert_count"]
    .fillna(0.0)
    .resample("15min")
    .sum()
)

price_15m = price_15m.loc[price_15m.index >= START_UTC]
alert_15m = alert_15m.reindex(price_15m.index).fillna(0.0)

print("START_UTC:", START_UTC)
print("price_15m rows:", len(price_15m), "alert_15m rows:", len(alert_15m))

In [ ]:
# Plot 15m candlestick + 15m alert + rolling correlation
ROLL_CORR_WINDOW = 96  # 96 * 15m ~= 1 day

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1, figsize=(18, 12), sharex=True,
    gridspec_kw={"height_ratios": [3, 1, 1], "hspace": 0.12}
)

x = mdates.date2num(price_15m.index.to_pydatetime())
bar_width = (15 / (24 * 60)) * 0.8

# Candlestick
for xi, o, h, l, c in zip(x, price_15m["open"], price_15m["high"], price_15m["low"], price_15m["close"]):
    color = "#16a34a" if c >= o else "#dc2626"
    ax1.vlines(xi, l, h, color=color, linewidth=0.7, alpha=0.9)
    body_low = min(o, c)
    body_h = max(abs(c - o), 1e-6)
    ax1.add_patch(Rectangle((xi - bar_width / 2, body_low), bar_width, body_h, facecolor=color, edgecolor=color, alpha=0.9))

ax1.set_title("BTCUSDT 15m Candlestick + Whale Alert Count (from merged table)")
ax1.set_ylabel("Price (USD)")
ax1.grid(alpha=0.2, linestyle="--")

# Alert count
ax2.bar(price_15m.index, alert_15m.values, width=bar_width, color="#2563eb", alpha=0.9)
ax2.set_ylabel("Alert Count")
ax2.grid(alpha=0.2, linestyle="--", axis="y")

# Rolling correlation: |return| vs alert count
abs_ret_15m = price_15m["close"].pct_change().abs()
rolling_corr = abs_ret_15m.rolling(ROLL_CORR_WINDOW, min_periods=ROLL_CORR_WINDOW).corr(alert_15m)

ax3.plot(price_15m.index, rolling_corr, color="#7c3aed", linewidth=1.3)
ax3.axhline(0.0, color="#6b7280", linewidth=0.9, linestyle="--")
ax3.set_ylim(-0.1, 0.6)
ax3.set_ylabel("Rolling Corr")
ax3.set_title("Correlation between BTCUSDT 15m OHLC and Whale Signal Density")
ax3.grid(alpha=0.2, linestyle="--")
ax3.set_xlabel("UTC Time")

ax3.xaxis.set_major_locator(mdates.AutoDateLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.setp(ax3.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# Regime validation (low / mid / high) — only rows between first and last whale alert minute
_wac = pd.to_numeric(df["whale_alert_count"], errors="coerce").fillna(0)
if not (_wac > 0).any():
    raise ValueError("No rows with whale_alert_count > 0")
WHALE_ALERT_FIRST = df.loc[_wac > 0, "time_utc"].min()
WHALE_ALERT_LAST = df.loc[_wac > 0, "time_utc"].max()

reg = pd.DataFrame(index=price_15m.index)
reg["close"] = price_15m["close"]
reg["alert_15m"] = alert_15m
reg["ret_15m"] = reg["close"].pct_change()
reg["vol_1d"] = reg["ret_15m"].rolling(96, min_periods=96).std()

reg = reg.loc[(reg.index >= WHALE_ALERT_FIRST) & (reg.index <= WHALE_ALERT_LAST)].copy()
q1, q2 = reg["vol_1d"].quantile([1 / 3, 2 / 3])
reg["vol_regime"] = np.nan
reg.loc[reg["vol_1d"] <= q1, "vol_regime"] = 0
reg.loc[(reg["vol_1d"] > q1) & (reg["vol_1d"] <= q2), "vol_regime"] = 1
reg.loc[reg["vol_1d"] > q2, "vol_regime"] = 2
reg = reg.dropna(subset=["vol_regime"]).copy()
reg["vol_regime"] = reg["vol_regime"].astype(int)

def _mean_alert_nonzero(s):
    nz = s[s > 0]
    return nz.mean() if len(nz) else np.nan

def _median_alert_nonzero(s):
    nz = s[s > 0]
    return nz.median() if len(nz) else np.nan

def _p90_alert_nonzero(s):
    nz = s[s > 0]
    return nz.quantile(0.90) if len(nz) else np.nan

# mean / median / p90: only rows with alert > 0 (sparse zeros excluded)
summary = (
    reg.groupby("vol_regime")["alert_15m"]
    .agg(
        count="size",
        n_nonzero=lambda s: (s > 0).sum(),
        mean_alert=_mean_alert_nonzero,
        median_alert=_median_alert_nonzero,
        max_alert="max",
    )
)
summary["p90_alert"] = reg.groupby("vol_regime")["alert_15m"].apply(_p90_alert_nonzero)
summary["nonzero_rate_%"] = reg.groupby("vol_regime")["alert_15m"].apply(lambda s: (s > 0).mean() * 100)
summary["avg_abs_ret_bp"] = reg.groupby("vol_regime")["ret_15m"].apply(lambda s: s.abs().mean() * 1e4)

summary_show = summary.reindex([0, 1, 2]).copy()
summary_show.index = pd.Index(["low", "mid", "high"], name="vol_regime")

print("Vol regime window (first→last whale alert minute):", WHALE_ALERT_FIRST, "->", WHALE_ALERT_LAST)
print("vol terciles:", float(q1), float(q2))
display(summary_show.round(3))

## Daily alert charts on full whale-data days only

Rules:
- Start from the first day where whale data appears.
- End at the last day where whale data appears.
- Exclude boundary partial days (if first/last day is not complete).

In [ ]:
# Daily active alert minutes + daily alert notional amount
w = df.copy()

# Whale start/end from actual whale timestamp availability
if "minute_utc" in w.columns:
    w_whale = w.dropna(subset=["minute_utc"]).copy()
    whale_start = w_whale["minute_utc"].min()
    whale_end = w_whale["minute_utc"].max()
else:
    raise ValueError("minute_utc column not found in merged table.")

if pd.isna(whale_start) or pd.isna(whale_end):
    raise ValueError("No whale timestamp found in merged table.")

# Remove boundary partial days
full_start = whale_start.floor("D") if whale_start == whale_start.floor("D") else whale_start.floor("D") + pd.Timedelta(days=1)
full_end = whale_end.floor("D") if whale_end.time() == pd.Timestamp("23:59:00").time() else whale_end.floor("D") - pd.Timedelta(days=1)

if full_end < full_start:
    raise ValueError("No full whale-data days available after excluding partial boundary days.")

mask = (w["time_utc"] >= full_start) & (w["time_utc"] <= (full_end + pd.Timedelta(hours=23, minutes=59)))
wd = w.loc[mask].copy().set_index("time_utc").sort_index()

wd["whale_alert_count"] = pd.to_numeric(wd.get("whale_alert_count"), errors="coerce").fillna(0.0)
wd["whale_alert_notional_abs"] = pd.to_numeric(wd.get("whale_alert_notional_abs"), errors="coerce").fillna(0.0)

daily_active_alert_minutes = (wd["whale_alert_count"] > 0).resample("1D").sum()
daily_alert_notional_usd = wd["whale_alert_notional_abs"].resample("1D").sum()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

axes[0].bar(daily_active_alert_minutes.index, daily_active_alert_minutes.values, color="#2563eb", alpha=0.9)
axes[0].set_title("Daily Whale Alert Activity (Number of Active Minutes, UTC)")
axes[0].set_ylabel("minutes")
axes[0].grid(alpha=0.25, linestyle="--", axis="y")

axes[1].plot(daily_alert_notional_usd.index, daily_alert_notional_usd.values, color="#dc2626", linewidth=1.6)
axes[1].set_title("Daily Whale Alert Notional (USD, Absolute Sum of Long + Short Alerts)")
axes[1].set_ylabel("USD")
axes[1].grid(alpha=0.25, linestyle="--")

axes[1].xaxis.set_major_locator(mdates.AutoDateLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.show()

print("Whale raw range:", whale_start, "->", whale_end)
print("Full-day range used:", full_start, "->", full_end)
print("Days plotted:", len(daily_active_alert_minutes))

## 1m Regime Validation (for comparison)

Below we run the same regime idea on **1-minute** data:
- `ret_1m = close.pct_change()`
- `vol_1d_1m = rolling std of ret_1m over 1440 bars` (about 1 day)
- split into terciles: `low / mid / high` — **terciles and summary use only minutes from the first whale alert through the last whale alert** (same window as the 15m table above).
- **mean, median, and p90 of alert count** are computed **only on bars with** `alert > 0` **within each regime** (see `n_nonzero`); `count` is still all bars in that regime.
- compute the same alert statistics and compare with 15m table.

In [ ]:
# 1m regime table (same whale-alert window as 15m regime cell)
if "WHALE_ALERT_FIRST" not in globals() or "WHALE_ALERT_LAST" not in globals():
    _wac = pd.to_numeric(df["whale_alert_count"], errors="coerce").fillna(0)
    if not (_wac > 0).any():
        raise ValueError("No rows with whale_alert_count > 0")
    WHALE_ALERT_FIRST = df.loc[_wac > 0, "time_utc"].min()
    WHALE_ALERT_LAST = df.loc[_wac > 0, "time_utc"].max()

reg1 = df[["time_utc", "close", "whale_alert_count"]].copy()
reg1["close"] = pd.to_numeric(reg1["close"], errors="coerce")
reg1["alert_1m"] = pd.to_numeric(reg1["whale_alert_count"], errors="coerce").fillna(0.0)
reg1 = reg1.drop(columns=["whale_alert_count"]).dropna(subset=["time_utc", "close"])
reg1 = reg1.set_index("time_utc").sort_index()
reg1["ret_1m"] = reg1["close"].pct_change()
reg1["vol_1d_1m"] = reg1["ret_1m"].rolling(1440, min_periods=1440).std()

reg1 = reg1.loc[(reg1.index >= WHALE_ALERT_FIRST) & (reg1.index <= WHALE_ALERT_LAST)].copy()
q1_1m, q2_1m = reg1["vol_1d_1m"].quantile([1 / 3, 2 / 3])
reg1["vol_regime"] = np.nan
reg1.loc[reg1["vol_1d_1m"] <= q1_1m, "vol_regime"] = 0
reg1.loc[(reg1["vol_1d_1m"] > q1_1m) & (reg1["vol_1d_1m"] <= q2_1m), "vol_regime"] = 1
reg1.loc[reg1["vol_1d_1m"] > q2_1m, "vol_regime"] = 2
reg1 = reg1.dropna(subset=["vol_regime"]).copy()
reg1["vol_regime"] = reg1["vol_regime"].astype(int)

def _mean_alert_nonzero(s):
    nz = s[s > 0]
    return nz.mean() if len(nz) else np.nan

def _median_alert_nonzero(s):
    nz = s[s > 0]
    return nz.median() if len(nz) else np.nan

def _p90_alert_nonzero(s):
    nz = s[s > 0]
    return nz.quantile(0.90) if len(nz) else np.nan

# mean / median / p90: only rows with alert > 0 (sparse zeros excluded)
summary_1m = (
    reg1.groupby("vol_regime")["alert_1m"]
    .agg(
        count="size",
        n_nonzero=lambda s: (s > 0).sum(),
        mean_alert=_mean_alert_nonzero,
        median_alert=_median_alert_nonzero,
        max_alert="max",
    )
)
summary_1m["p90_alert"] = reg1.groupby("vol_regime")["alert_1m"].apply(_p90_alert_nonzero)
summary_1m["nonzero_rate_%"] = reg1.groupby("vol_regime")["alert_1m"].apply(lambda s: (s > 0).mean() * 100)
summary_1m["avg_abs_ret_bp"] = reg1.groupby("vol_regime")["ret_1m"].apply(lambda s: s.abs().mean() * 1e4)
summary_1m = summary_1m.reindex([0, 1, 2])
summary_1m.index = pd.Index(["low", "mid", "high"], name="vol_regime")

print("Vol regime window (first→last whale alert minute):", WHALE_ALERT_FIRST, "->", WHALE_ALERT_LAST)
print("1m vol terciles:", float(q1_1m), float(q2_1m))
display(summary_1m.round(3))

# Optional side-by-side compare with existing 15m summary
if "summary_show" in globals():
    s15 = summary_show.copy()
    s15["timeframe"] = "15m"
    s1 = summary_1m.copy()
    s1["timeframe"] = "1m"

    cmp = pd.concat([s15.reset_index(), s1.reset_index()], axis=0, ignore_index=True)
    cmp = cmp[
        [
            "timeframe",
            "vol_regime",
            "count",
            "n_nonzero",
            "mean_alert",
            "median_alert",
            "p90_alert",
            "nonzero_rate_%",
            "avg_abs_ret_bp",
        ]
    ]
    display(cmp.round(3))